# 05 — Đánh giá Test, phân tích lỗi và triển khai
Chỉ đọc bundle/cache từ chuỗi notebook 02–04 cùng run_id. Không fallback sang production.
Mô hình và threshold đã chốt trên Validation; kết quả Test không dùng để chọn lại.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT=Path.cwd().resolve()
if not (PROJECT_ROOT/'src').is_dir(): PROJECT_ROOT=PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from src.config import NOTEBOOK_SAVED_MODELS_DIR,NOTEBOOK_METRICS_DIR,NOTEBOOK_FIGURES_DIR,PROCESSED_DATA_DIR,FBETA_BETA
from src.models.persistence import load_spam_pipeline,save_spam_pipeline
from src.evaluation.metrics import SpamModelEvaluator
from src.evaluation.error_analysis import extract_misclassified_samples,profile_error_patterns
from src.evaluation.visualization import plot_confusion_matrix_heatmap,plot_roc_and_pr_curves,plot_models_comparison_bar
from src.deployment.predictor import SpamInferenceService
for directory in (NOTEBOOK_METRICS_DIR,NOTEBOOK_FIGURES_DIR): directory.mkdir(parents=True,exist_ok=True)
pipeline_path=NOTEBOOK_SAVED_MODELS_DIR/'spam_pipeline.joblib'
bundle=load_spam_pipeline(pipeline_path)
assert bundle['run_id']==(PROCESSED_DATA_DIR/'notebook_run_id.txt').read_text(encoding='utf-8').strip(),'Chạy lại 02–04.'
X_test=sp.load_npz(PROCESSED_DATA_DIR/'X_test.npz'); y_test=np.load(PROCESSED_DATA_DIR/'y_test.npy')
test_df=pd.read_csv(PROCESSED_DATA_DIR/'test_clean.csv',keep_default_na=False)
assert test_df.source_id.to_list()==bundle['source_ids']['test'],'Thứ tự dữ liệu Test không khớp artifacts.'
assert np.array_equal((test_df.label=='spam').astype(int).to_numpy(),y_test)
evaluator=SpamModelEvaluator()
model=bundle['model']; best_threshold=bundle['best_threshold']

from hashlib import sha256
if not bundle.get('cache_sha256'):
    raise RuntimeError('Cache thiếu manifest: chạy lại notebook 02–04.')
for filename, expected_hash in bundle['cache_sha256'].items():
    actual_hash=sha256((PROCESSED_DATA_DIR/filename).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f'Cache không khớp artifacts: {filename}; chạy lại notebook 02–04.')
print('PASS: hash ma trận, nhãn và CSV khớp artifacts.')


PASS: hash ma trận, nhãn và CSV khớp artifacts.


## 1. Tính lại đầy đủ metrics và biểu đồ từ Test
Bảng so sánh dùng threshold đã chọn riêng cho mỗi ứng viên trên Validation (Hard Voting dùng 0.5).
Winner vẫn là lựa chọn ở notebook 04, kể cả khi ứng viên khác có Test F1 cao hơn.


In [2]:
results={}
for name,candidate in bundle['candidate_models'].items():
    scores=candidate.predict_proba(X_test)[:,1]
    predictions=(scores>=bundle['candidate_thresholds'][name]).astype(int)
    results[name]=evaluator.compute_all_metrics(y_test,predictions,scores,beta=FBETA_BETA)
summary=evaluator.build_comparison_summary(results)
summary['selected_on_validation']=summary['Mô hình']==bundle['model_name']
summary['run_id']=bundle['run_id']
display(summary)
summary.to_csv(NOTEBOOK_METRICS_DIR/'test_models_comparison.csv',index=False)
scores=model.predict_proba(X_test)[:,1]
predictions=(scores>=best_threshold).astype(int)
final_metrics=results[bundle['model_name']]
plot_confusion_matrix_heatmap(final_metrics['confusion_matrix'],model_name=bundle['model_name'],save_path=NOTEBOOK_FIGURES_DIR/'confusion_matrix.png')
plot_roc_and_pr_curves(y_test,scores,best_threshold=best_threshold,model_name=bundle['model_name'],save_path=NOTEBOOK_FIGURES_DIR/'roc_pr_curve.png')
plot_models_comparison_bar(summary,save_path=NOTEBOOK_FIGURES_DIR/'models_comparison.png')
audit=evaluator.audit_evaluation_consistency(y_test,predictions,scores)
display(audit)
assert (audit.Status=='PASS').all()
bundle['test_metrics']=final_metrics
save_spam_pipeline(bundle,pipeline_path)


,Mô hình,Accuracy,Precision,Recall,F1-Score,F-Beta,ROC-AUC,selected_on_validation,run_id
0,Soft Voting,0.675052,0.579399,0.871671,0.696101,0.791789,0.761489,True,64dc96d1a13d4cd9a312f4f22cbb892c
1,Logistic Regression,0.658511,0.564718,0.873285,0.685895,0.787253,0.738143,False,64dc96d1a13d4cd9a312f4f22cbb892c
2,Logistic Regression (Tuned),0.654721,0.562139,0.865214,0.681500,0.781000,0.729165,False,64dc96d1a13d4cd9a312f4f22cbb892c
3,Linear SVM (Tuned),0.644383,0.553213,0.868442,0.675879,0.779597,0.713279,False,64dc96d1a13d4cd9a312f4f22cbb892c
4,Complement Naive Bayes (Tuned),0.647140,0.556848,0.849879,0.672843,0.768950,0.752193,False,64dc96d1a13d4cd9a312f4f22cbb892c
5,Complement Naive Bayes,0.645762,0.555614,0.850686,0.672194,0.769006,0.752167,False,64dc96d1a13d4cd9a312f4f22cbb892c
6,Linear SVM,0.606134,0.522222,0.910412,0.663725,0.792580,0.688032,False,64dc96d1a13d4cd9a312f4f22cbb892c
7,Hard Voting,0.669883,0.595254,0.708636,0.647015,0.682631,0.710972,False,64dc96d1a13d4cd9a312f4f22cbb892c


,Audit Item,Observed,Expected,Status
0,Total Sample Sum Conservation (TP+FP+TN+FN == N),2902.00000,2902,PASS
1,Accuracy Mathematical Consistency,0.67505,0.67505,PASS
2,"Precision in [0, 1] Bounds",0.57940,0.0 <= P <= 1.0,PASS
3,"Recall in [0, 1] Bounds",0.87170,0.0 <= R <= 1.0,PASS
4,"F1-Score in [0, 1] Bounds",0.69610,0.0 <= F1 <= 1.0,PASS
5,ROC-AUC Valid (if available),0.76150,>= 0.50,PASS


WindowsPath('E:/Hoc_May_Tren_truong/lab_1 spam_classifier/notebooks/results/saved_models/spam_pipeline.joblib')

## 2. Trích xuất FP/FN và chẩn đoán định tính
Tự tạo CSV từ dự đoán hiện tại; nguyên nhân từ khóa chỉ là giả thuyết, không phải giải thích nhân quả của mô hình.


In [3]:
fp,fn=extract_misclassified_samples(test_df,y_test,predictions,scores)
errors=profile_error_patterns(fp,fn)
if errors.empty:
    errors=pd.concat([fp,fn],ignore_index=True)
    errors['suspected_cause']=pd.Series(dtype=str)
errors['run_id']=bundle['run_id']
errors.to_csv(NOTEBOOK_METRICS_DIR/'misclassified_samples.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame([{'type':'FP','count':len(fp)},{'type':'FN','count':len(fn)}]))
for kind in ('False Positive','False Negative'):
    subset=errors[errors.error_type.str.contains(kind,na=False)]
    display(subset[['source_id','text','spam_probability','suspected_cause']].head(5))
    if len(subset):
        print(kind,'số từ trung bình:',subset.text.str.split().str.len().mean())
        print(subset.suspected_cause.value_counts().head(5))
print(f"Winner đã chốt: {bundle['model_name']}; Test FP={len(fp)}, FN={len(fn)}, Recall={final_metrics['recall']:.4f}, F1={final_metrics['f1']:.4f}.")
print('Không dùng phân tích lỗi Test để chọn lại ngưỡng hoặc mô hình trong lần đánh giá này.')


,type,count
0,FP,784
1,FN,159


,source_id,text,spam_probability,suspected_cause
0,39726,"louise , i worked with the team this morning a...",0.5081,Dính từ khóa nhạy cảm (call)
1,19761,. us email international business charset = is...,0.5571,Dính từ khóa nhạy cảm (offer); Dấu chấm than (4)
2,17310,claim a free escapenumber victoria's secret gi...,0.7707,"Dính từ khóa nhạy cảm (offer, claim, free)"
3,19508,make use of the opportunity escapenumber anatr...,0.3986,Đặc trưng từ vựng trùng lặp với tập Spam
4,24996,the only clinically tested penis en _ largemen...,0.7122,Dấu chấm than (2)


False Positive số từ trung bình: 240.25255102040816
suspected_cause
Đặc trưng từ vựng trùng lặp với tập Spam    511
Dính từ khóa nhạy cảm (free)                 49
Dính từ khóa nhạy cảm (call)                 37
Dính từ khóa nhạy cảm (money)                32
Dính từ khóa nhạy cảm (offer)                19
Name: count, dtype: int64


,source_id,text,spam_probability,suspected_cause
784,25504,can you have someone send me the list that you...,0.2891,"Spam tinh vi, không dùng từ khóa lộ liễu"
785,6158,thesmszone.com lets you send free anonymous an...,0.3555,Xác suất spam dưới ngưỡng quyết định
786,33371,spot young want power read next space gold fig...,0.3250,"Spam tinh vi, không dùng từ khóa lộ liễu"
787,27233,agree according to one of our articles 'that t...,0.3746,"Spam tinh vi, không dùng từ khóa lộ liễu"
788,21338,on wed may escapenumber escapenumber at escape...,0.3785,Xác suất spam dưới ngưỡng quyết định


False Negative số từ trung bình: 281.9496855345912
suspected_cause
Spam tinh vi, không dùng từ khóa lộ liễu                                   103
Xác suất spam dưới ngưỡng quyết định                                        54
Email quá ngắn, thiếu từ khóa; Spam tinh vi, không dùng từ khóa lộ liễu      2
Name: count, dtype: int64
Winner đã chốt: Soft Voting; Test FP=784, FN=159, Recall=0.8717, F1=0.6961.
Không dùng phân tích lỗi Test để chọn lại ngưỡng hoặc mô hình trong lần đánh giá này.


## 3. Kiểm tra parity sau lưu/nạp và demo email mới
Đối chiếu cache với đặc trưng tái tạo trên 10 mẫu Test; xác suất và quyết định phải khớp trước/sau lưu.


In [4]:
service=SpamInferenceService(bundle_path=pipeline_path).load_service()
sample=test_df.head(10)
rebuilt=service.feature_builder.transform(sample).to_scipy_csr()[:,bundle['selected_indices']]
np.testing.assert_allclose(rebuilt.toarray(),X_test[:len(sample)].toarray(),atol=1e-12)
for i,text in enumerate(sample.text):
    result=service.predict_single(text)
    assert result['spam_probability']==round(float(scores[i]),4)
    assert result['is_spam']==bool(predictions[i])
print('PASS: cache, thứ tự Top-K và suy luận sau lưu/nạp nhất quán.')
demo_emails=[
 'URGENT! You won a cash prize! Call now to claim your award!',
 'Dear team, please review the weekly project report and meeting notes.',
 'Free entry to win tickets! Text now to claim your prize!',
 'Hey, are we meeting for lunch today?',
 'Your account statement is ready. Please sign in through the official website.',
]
display(service.predict_batch(demo_emails))
print('Demo minh họa, không phải tập kiểm thử có nhãn. Tỷ lệ phiếu Hard Voting và sigmoid SVM chưa được calibration.')


PASS: cache, thứ tự Top-K và suy luận sau lưu/nạp nhất quán.


,raw_text,prediction,is_spam,spam_probability,threshold_used,risk_level,detected_signals
0,URGENT! You won a cash prize! Call now to clai...,SPAM,True,0.7027,0.39,Cao,"[Từ khóa nhạy cảm: call, cash, claim, prize, u..."
1,"Dear team, please review the weekly project re...",NOT SPAM / HAM,False,0.3046,0.39,Trung bình,[]
2,Free entry to win tickets! Text now to claim y...,SPAM,True,0.7118,0.39,Cao,"[Từ khóa nhạy cảm: claim, free, prize, win, Dấ..."
3,"Hey, are we meeting for lunch today?",NOT SPAM / HAM,False,0.2674,0.39,Trung bình,[]
4,Your account statement is ready. Please sign i...,SPAM,True,0.4150,0.39,Cao,[]


Demo minh họa, không phải tập kiểm thử có nhãn. Tỷ lệ phiếu Hard Voting và sigmoid SVM chưa được calibration.
